# 02 — Exploratory Data Analysis

Цель ноутбука: проверить пропуски, распределения, корреляции и связи между ключевыми признаками и success label.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_SYNTHETIC = ROOT / "data" / "synthetic"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

print("Project root:", ROOT)
print("Synthetic data:", DATA_SYNTHETIC)

In [ ]:
employees = pd.read_csv(DATA_SYNTHETIC / "employees.csv")
tasks = pd.read_csv(DATA_SYNTHETIC / "tasks.csv")
assignments = pd.read_csv(DATA_SYNTHETIC / "assignments.csv")

merged = assignments.merge(tasks, on="task_id", suffixes=("_assignment", "_task"))
merged = merged.merge(employees, on="employee_id", suffixes=("", "_employee"))

print("merged:", merged.shape)

In [ ]:
missing_report = pd.DataFrame(
    {
        "employees_missing": employees.isna().sum(),
        "tasks_missing": tasks.isna().sum(),
        "assignments_missing": assignments.isna().sum(),
    }
).fillna(0)

display(missing_report.sort_values("assignments_missing", ascending=False).head(30))

In [ ]:
numeric_cols = [
    "skill_match_score",
    "growth_match_score",
    "risk_score",
    "quality_score",
    "employee_workload_at_assignment",
    "delay_days",
    "success_probability",
    "success_label",
]

corr = assignments[numeric_cols].corr(numeric_only=True)
display(corr)

In [ ]:
fig = px.imshow(corr, text_auto=True, title="Correlation matrix")
fig.show()
fig.write_html(FIGURES / "eda_correlation_matrix.html")

In [ ]:
success_by_skill = assignments.groupby("success_label")["skill_match_score"].mean().reset_index()

fig = px.bar(
    success_by_skill,
    x="success_label",
    y="skill_match_score",
    title="Average skill match by success label",
)
fig.show()
fig.write_html(FIGURES / "eda_skill_match_by_success.html")

In [ ]:
success_by_workload = (
    assignments.groupby("success_label")["employee_workload_at_assignment"]
    .mean()
    .reset_index()
)

fig = px.bar(
    success_by_workload,
    x="success_label",
    y="employee_workload_at_assignment",
    title="Average workload by success label",
)
fig.show()
fig.write_html(FIGURES / "eda_workload_by_success.html")

In [ ]:
complexity_success = merged.groupby("complexity")["success_label"].mean().reset_index()

fig = px.line(
    complexity_success,
    x="complexity",
    y="success_label",
    markers=True,
    title="Success rate by task complexity",
)
fig.show()
fig.write_html(FIGURES / "eda_success_by_complexity.html")